In [206]:
import jams
import numpy as np
import os
import pretty_midi
from music21 import stream, note, tempo, meter, clef, articulations, expressions, spanner, interval, instrument
from music21 import note as m21_note  # Ensure we have access to Rest
import xml.etree.ElementTree as ET
import random
np.int = int # deprecated np.int
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import math

In [207]:
midi_dir= '/data/akshaj/MusicAI/GuitarSet/MIDIAnnotations/'
midi_path = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

In [208]:
# Standard guitar tuning (MIDI pitch for each open string)
STANDARD_TUNING = {
    6: 40,  # E2 (low E)
    5: 45,  # A2
    4: 50,  # D3
    3: 55,  # G3
    2: 59,  # B3
    1: 64   # E4 (high e)
}

In [209]:
def midi_pitch_to_guitar_positions(midi_pitch, tuning=STANDARD_TUNING, max_fret=12):
    """
    Find all possible (string, fret) positions for a given MIDI pitch.
    
    Args:
        midi_pitch: MIDI note number (e.g., 60 = middle C)
        tuning: Dictionary of {string_number: open_string_midi_pitch}
        max_fret: Maximum fret to consider
    
    Returns:
        List of (string, fret) tuples, sorted by preference
    """
    positions = []
    
    for string_num in range(1, 7):  # Strings 1-6
        open_pitch = tuning[string_num]
        fret = midi_pitch - open_pitch
        
        # Valid if fret is 0-12 (or your max_fret)
        if 0 <= fret <= max_fret:
            positions.append((string_num, fret))
    
    # Sort by preference: middle strings first, lower frets preferred
    # This creates more natural fingering
    def position_score(pos):
        string, fret = pos
        # Prefer middle strings (3, 4) and lower frets
        string_penalty = abs(string - 3.5)  # Prefer strings 3-4
        fret_penalty = fret * 0.1  # Slight preference for lower frets
        return string_penalty + fret_penalty
    
    positions.sort(key=position_score)
    return positions

In [210]:
def choose_best_position(midi_pitch, previous_position=None, tuning=STANDARD_TUNING):
    """
    Choose the best string/fret position for a pitch.
    Takes into account the previous position to minimize hand movement.
    
    Args:
        midi_pitch: MIDI note number
        previous_position: Previous (string, fret) tuple, or None
        tuning: Guitar tuning
    
    Returns:
        (string, fret) tuple
    """
    positions = midi_pitch_to_guitar_positions(midi_pitch, tuning)
    
    if not positions:
        # Pitch out of range - use closest approximation
        # Find the closest string
        closest_string = min(tuning.keys(), 
                           key=lambda s: abs(tuning[s] - midi_pitch))
        fret = max(0, min(12, midi_pitch - tuning[closest_string]))
        return (closest_string, fret)
    
    if previous_position is None:
        # No previous position - use most natural position
        return positions[0]
    
    # Choose position closest to previous position (minimize hand movement)
    prev_string, prev_fret = previous_position
    
    def distance_score(pos):
        string, fret = pos
        string_distance = abs(string - prev_string)
        fret_distance = abs(fret - prev_fret)
        # Weight fret distance more (moving along frets is harder than changing strings)
        return fret_distance * 2 + string_distance
    
    return min(positions, key=distance_score)

In [211]:
def midi_to_jams_with_tablature(midi_path, tuning=STANDARD_TUNING):
    """
    Convert MIDI file to JAMS with intelligent tablature mapping.
    Extracts tempo and time signature directly from the MIDI file.
    
    Args:
        midi_path: Path to MIDI file
        tuning: Guitar tuning dictionary
    
    Returns:
        JAMS object with both note_midi and tab_note annotations,
        plus extracted tempo_bpm and time_signature
    """
    # Load MIDI
    pm = pretty_midi.PrettyMIDI(midi_path)
    guitar_notes = pm.instruments[0].notes
    
    # Extract tempo from MIDI
    # get_tempo_changes() returns (times, tempos) arrays
    tempo_times, tempos = pm.get_tempo_changes()
    if len(tempos) > 0:
        tempo_bpm = tempos[0]  # Use first tempo (or could average/use most common)
    else:
        tempo_bpm = 120  # Default fallback
    
    # Extract time signature from MIDI
    # time_signature_changes is a list of TimeSignature objects
    if pm.time_signature_changes:
        ts = pm.time_signature_changes[0]  # Use first time signature
        time_sig_numerator = ts.numerator
        time_sig_denominator = ts.denominator
        print(f"Time signature from MIDI: {time_sig_numerator}/{time_sig_denominator}")
    else:
        print("No time signature found, using default 4/4 instead")
        time_sig_numerator = 4  # Default 4/4
        time_sig_denominator = 4
    
    time_signature = f"{time_sig_numerator}/{time_sig_denominator}"
    
    print(f"Extracted from MIDI - Tempo: {tempo_bpm:.1f} BPM, Time Signature: {time_signature}")
    
    # Create JAMS
    jam = jams.JAMS()
    
    # Store tempo and time signature in JAMS file metadata
    jam.file_metadata.duration = pm.get_end_time()
    jam.sandbox.tempo_bpm = tempo_bpm
    jam.sandbox.time_signature = time_signature
    jam.sandbox.time_sig_numerator = time_sig_numerator
    jam.sandbox.time_sig_denominator = time_sig_denominator
    
    # Add note_midi annotation
    note_ann = jams.Annotation(namespace='note_midi')
    for n in guitar_notes:
        note_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=n.pitch,
            confidence=n.velocity / 127
        )
    jam.annotations.append(note_ann)
    
    # Add tab_note annotation with intelligent string/fret mapping
    tab_ann = jams.Annotation(namespace='tab_note')
    
    previous_position = None
    for n in guitar_notes:
        # Find best position for this pitch
        string, fret = choose_best_position(n.pitch, previous_position, tuning)
        previous_position = (string, fret)
        
        value = {
            "pitch": n.pitch,
            "string": string,
            "fret": fret,
            "techniques": []  # No techniques from MIDI
        }
        
        tab_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=value,
            confidence=n.velocity / 127
        )
    
    jam.annotations.append(tab_ann)
    
    print(f"Converted {len(guitar_notes)} notes to tablature")
    print(f"Pitch range: {min(n.pitch for n in guitar_notes)} - {max(n.pitch for n in guitar_notes)}")
    
    return jam

In [212]:
def post_process_musicxml(xml_path, technique_map=None):
    """
    Post-process MusicXML to add proper Guitar Pro compatible elements.
    
    Args:
        xml_path: Path to the MusicXML file
        technique_map: Dict mapping note indices to technique info
                      {note_index: {'technique': 'hammer-on'|'pull-off', 'is_target': bool}}
    """
    from lxml import etree
    
    if technique_map is None:
        technique_map = {}
    
    tree = etree.parse(xml_path)
    root = tree.getroot()
    
    # Collect all notes in order across all measures
    all_notes = []
    for measure in root.iter('measure'):
        for elem in measure:
            if elem.tag == 'note':
                # Skip grace notes and chord notes (notes that are part of a chord)
                # Chord notes have a <chord/> element
                chord_elem = elem.find('chord')
                if chord_elem is None:
                    all_notes.append(elem)
    
    print(f"  Found {len(all_notes)} notes for technique processing")
    print(f"  Technique map: {technique_map}")
    
    # Track slur numbers to avoid conflicts
    slur_number = 1
    
    # Process techniques - add hammer-on/pull-off and slur elements
    for note_idx, tech_info in technique_map.items():
        if note_idx >= len(all_notes) or note_idx < 1:
            print(f"  Warning: Invalid note index {note_idx}")
            continue
        
        technique = tech_info['technique']
        
        # The target note (where the technique annotation goes)
        target_note = all_notes[note_idx]
        # The source note (previous note, where slur starts)
        source_note = all_notes[note_idx - 1]
        
        print(f"  Adding {technique} slur from note {note_idx-1} to note {note_idx}")
        
        # Get or create notations element for source note
        source_notations = source_note.find('notations')
        if source_notations is None:
            source_notations = etree.SubElement(source_note, 'notations')
        
        # Get or create technical element for source note
        source_technical = source_notations.find('technical')
        if source_technical is None:
            source_technical = etree.SubElement(source_notations, 'technical')
        
        # Get or create notations element for target note
        target_notations = target_note.find('notations')
        if target_notations is None:
            target_notations = etree.SubElement(target_note, 'notations')
        
        # Get or create technical element for target note
        target_technical = target_notations.find('technical')
        if target_technical is None:
            target_technical = etree.SubElement(target_notations, 'technical')
        
        # Determine element name based on technique
        if technique == 'hammer-on':
            tech_elem_name = 'hammer-on'
            tech_text = 'H'
        else:  # pull-off
            tech_elem_name = 'pull-off'
            tech_text = 'P'
        
        # Add technique start element to source note
        tech_start = etree.Element(tech_elem_name)
        tech_start.set('number', str(slur_number))
        tech_start.set('type', 'start')
        tech_start.text = tech_text
        source_technical.insert(0, tech_start)  # Insert at beginning
        
        # Add slur start to source note
        slur_start = etree.Element('slur')
        slur_start.set('number', str(slur_number))
        slur_start.set('type', 'start')
        slur_start.set('placement', 'above')
        source_notations.append(slur_start)
        
        # Add technique stop element to target note
        tech_stop = etree.Element(tech_elem_name)
        tech_stop.set('number', str(slur_number))
        tech_stop.set('type', 'stop')
        target_technical.append(tech_stop)
        
        # Add slur stop to target note
        slur_stop = etree.Element('slur')
        slur_stop.set('number', str(slur_number))
        slur_stop.set('type', 'stop')
        target_notations.append(slur_stop)
        
        slur_number += 1
    
    # Process other techniques (vibrato, harmonic, etc.)
    for measure in root.iter('measure'):
        elements = list(measure)
        measure[:] = []
        
        for elem in elements:
            if elem.tag == 'note':
                notations = elem.find('notations')
                has_vibrato = False
                
                if notations is not None:
                    technical = notations.find('technical')
                    if technical is not None:
                        # Check for harmonic
                        harmonic = technical.find('harmonic')
                        if harmonic is not None:
                            direction = etree.Element('direction')
                            direction.set('placement', 'above')
                            direction_type = etree.SubElement(direction, 'direction-type')
                            words = etree.SubElement(direction_type, 'words')
                            words.set('font-size', '10')
                            words.set('font-style', 'italic')
                            words.set('halign', 'center')
                            words.set('valign', 'top')
                            words.text = 'Harm.'
                            measure.append(direction)
                        
                        # Check for vibrato marker
                        fingering = technical.find('fingering')
                        if fingering is not None and fingering.text == 'vibrato':
                            technical.remove(fingering)
                            has_vibrato = True
                
                measure.append(elem)
                
                # Add Guitar Pro vibrato processing instruction after the note
                if has_vibrato:
                    vibrato_pi = etree.ProcessingInstruction('GP', '<root><vibrato type="Slight"/></root>')
                    elem.append(vibrato_pi)
            else:
                measure.append(elem)
    
    # Set guitar instrument
    for score_part in root.iter('score-part'):
        part_id = score_part.get('id')
        
        part_name = score_part.find('part-name')
        if part_name is None:
            part_name = etree.SubElement(score_part, 'part-name')
        part_name.text = 'Classical Guitar'
        
        part_abbrev = score_part.find('part-abbreviation')
        if part_abbrev is None:
            part_abbrev = etree.SubElement(score_part, 'part-abbreviation')
        part_abbrev.text = 'Guit.'
        
        score_inst = score_part.find('score-instrument')
        if score_inst is None:
            score_inst = etree.SubElement(score_part, 'score-instrument')
            score_inst.set('id', f'{part_id}-I1')
        
        inst_name = score_inst.find('instrument-name')
        if inst_name is None:
            inst_name = etree.SubElement(score_inst, 'instrument-name')
        inst_name.text = 'Classical Guitar'
        
        inst_sound = score_inst.find('instrument-sound')
        if inst_sound is None:
            inst_sound = etree.SubElement(score_inst, 'instrument-sound')
        inst_sound.text = 'pluck.guitar.nylon-string'
    
    tree.write(xml_path, encoding='utf-8', xml_declaration=True)
    print(f"✓ Post-processed {xml_path}")

In [213]:
def jams_to_musicxml_real(jam, output_xml='output.xml', tempo_bpm=120):
    """
    Convert JAMS with real tablature to MusicXML.
    Uses actual pitch information for correct notation.
    
    Args:
        jam: JAMS object with tab_note annotation
        output_xml: Output MusicXML file path
        tempo_bpm: Tempo in BPM
    
    Returns:
        Path to created XML file
    """
    from collections import defaultdict
    from music21 import chord as m21_chord
    
    # Find tab_note annotation
    tab_notes = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_notes = ann
            break
    
    if tab_notes is None:
        raise ValueError("No tab_note annotation found")
    
    print(f"Converting {len(tab_notes.data)} notes to MusicXML...")
    
    # Create score
    score = stream.Score()
    part = stream.Part()

    # Add instrument data
    guitar = instrument.Guitar()
    guitar.instrumentName = 'Classical Guitar (tablature)'
    guitar.instrumentAbbreviation = 'Guit.'
    guitar.instrumentSound = 'pluck.guitar.nylon-string'
    part.insert(0, guitar)
    
    # Add metadata
    part.insert(0, tempo.MetronomeMark(number=tempo_bpm))
    
    # Beat duration in seconds
    beat_sec = 60.0 / tempo_bpm
    
    # Setting correct time signature
    time_sig_numerator = getattr(jam.sandbox, 'time_sig_numerator', 4)
    time_sig_denominator = getattr(jam.sandbox, 'time_sig_denominator', 4)
    
    # Calculate beats per measure
    beats_per_measure = time_sig_numerator * (4.0 / time_sig_denominator)
    ts = meter.TimeSignature(f'{time_sig_numerator}/{time_sig_denominator}')
    part.insert(0, ts)
    part.insert(0, clef.TabClef())
    
    # Sort notes by time and collect quantized note data
    note_events = []
    for obs in sorted(tab_notes.data, key=lambda x: x.time):
        val = obs.value
        
        # Quantize time position to nearest eighth note
        time_beats = obs.time / beat_sec
        quantized_time = round(time_beats * 2) / 2
        
        # Quantize duration to standard values
        duration_beats = obs.duration / beat_sec
        quantized_duration = round(duration_beats * 2) / 2
        if quantized_duration < 0.5:
            quantized_duration = 0.5  # Minimum eighth note
        
        note_events.append({
            'time': quantized_time,
            'duration': quantized_duration,
            'pitch': val['pitch'],
            'string': val['string'],
            'fret': val['fret'],
            'techniques': val.get('techniques', [])
        })
    
    # Calculate total duration and number of complete measures needed
    if note_events:
        last_note_end = max(n['time'] + n['duration'] for n in note_events)
    else:
        last_note_end = 0
    
    total_measures = math.ceil(last_note_end / beats_per_measure)
    if total_measures < 1:
        total_measures = 1
    
    print(f"  Total duration: {last_note_end:.2f} beats -> {total_measures} measures")
    
    # Track note index for slur tracking (we'll handle slurs in post-processing)
    note_index = 0
    
    # Track techniques for post-processing
    # We'll store: {note_index: {'technique': 'hammer-on'|'pull-off', 'is_target': bool}}
    technique_map = {}
    
    for measure_num in range(total_measures):
        measure_start = measure_num * beats_per_measure
        measure_end = measure_start + beats_per_measure
        
        # Create a new measure stream
        current_measure = stream.Measure(number=measure_num + 1)
        
        # Get notes that START in this measure
        measure_notes = [n for n in note_events if measure_start <= n['time'] < measure_end]
        
        # Group notes by their start time (these are chords)
        notes_by_time = defaultdict(list)
        for n in measure_notes:
            # Convert to position within measure (0 to beats_per_measure)
            pos_in_measure = n['time'] - measure_start
            notes_by_time[pos_in_measure].append(n)

        sorted_positions = sorted(notes_by_time.keys())
        current_position = 0.0  # Track position within measure

        for pos_in_measure in sorted_positions:
            notes_at_time = notes_by_time[pos_in_measure]
            
            # Skip if this position is before our current position (overlapping notes)
            if pos_in_measure < current_position - 0.001:
                continue
            
            # Add rest if there's a gap
            if pos_in_measure > current_position + 0.001:
                rest_duration = pos_in_measure - current_position
                r = m21_note.Rest()
                r.quarterLength = rest_duration
                current_measure.append(r)
                current_position = pos_in_measure

            # Calculate available space in measure
            space_remaining = beats_per_measure - current_position
            
            # Determine the duration for this time slot
            min_duration = min(n['duration'] for n in notes_at_time)
            
            # Clamp duration to fit in measure
            actual_duration = min(min_duration, space_remaining)
            
            if actual_duration <= 0:
                continue  # No space left in measure

            # Create notes (as chord if multiple)
            if len(notes_at_time) == 1:
                note_event = notes_at_time[0]
                n = note.Note()
                n.pitch.midi = note_event['pitch']
                n.quarterLength = actual_duration
                n.editorial.stringNumber = note_event['string']
                n.editorial.fretNumber = note_event['fret']
                
                # Store technique info for post-processing
                techniques = note_event['techniques']
                if techniques:
                    for tech in techniques:
                        if not tech:
                            continue
                        
                        if tech in ['hammer-on', 'hammer_on']:
                            print(f"hammer-on detected at note {note_index}")
                            # Mark this note as the TARGET of a hammer-on
                            # The slur starts on the PREVIOUS note
                            technique_map[note_index] = {
                                'technique': 'hammer-on',
                                'is_target': True
                            }
                        
                        elif tech in ['pull-off', 'pull_off']:
                            print(f"pull-off detected at note {note_index}")
                            # Mark this note as the TARGET of a pull-off
                            technique_map[note_index] = {
                                'technique': 'pull-off', 
                                'is_target': True
                            }
                        
                        elif tech == 'bend':
                            print("bend detected")
                            bend_amount = 0.5
                            bend = articulations.FretBend(
                                bendAlter=interval.Interval(bend_amount)
                            )
                            n.articulations.append(bend)
                            n.expressions.append(expressions.TextExpression('B'))
                        
                        elif tech == 'slide':
                            print("slide detected")
                            n.expressions.append(expressions.TextExpression('/'))
                        
                        elif tech == 'vibrato':
                            print("Vibrato detected")
                            n.articulations.append(articulations.Fingering('vibrato'))
                        
                        elif tech == 'harmonic':
                            print("Harmonic detected")
                            n.articulations.append(articulations.Harmonic())
                
                current_measure.append(n)
                note_index += 1

            else:
                # Multiple notes at same time = chord
                chord_notes = []
                for note_event in notes_at_time:
                    n = note.Note()
                    n.pitch.midi = note_event['pitch']
                    n.editorial.stringNumber = note_event['string']
                    n.editorial.fretNumber = note_event['fret']
                    chord_notes.append(n)
                
                c = m21_chord.Chord(chord_notes)
                c.quarterLength = actual_duration
                current_measure.append(c)
                note_index += 1

            # Advance position
            current_position += actual_duration

        # Fill rest of measure with rest if needed
        if current_position < beats_per_measure - 0.001:
            remaining = beats_per_measure - current_position
            r = m21_note.Rest()
            r.quarterLength = remaining
            current_measure.append(r)
        
        # Verify measure duration before adding
        measure_duration = current_measure.duration.quarterLength
        if abs(measure_duration - beats_per_measure) > 0.01:
            print(f"  WARNING: Measure {measure_num + 1} has duration {measure_duration}, expected {beats_per_measure}")
        
        # Add measure to part
        part.append(current_measure)
    
    score.append(part)
    
    # Write to MusicXML
    try:
        score.write('musicxml', fp=output_xml)
        print(f"✓ Created {output_xml}")
    except Exception as e:
        print(f"Error: {e}")
        print("Trying with makeNotation...")
        score.write('musicxml', fp=output_xml, makeNotation=True)
        print(f"✓ Created {output_xml} (with notation fixes)")

    # Post-process the XML with technique information
    post_process_musicxml(output_xml, technique_map)
    
    return output_xml

In [214]:
# =============================================================================
# ADVANCED: Position optimization for better fingering
# =============================================================================

def optimize_tablature_positions(jam, max_stretch=4):
    """
    Optimize string/fret positions for better playability.
    Tries to keep notes within reasonable hand positions.
    
    Args:
        jam: JAMS object with tab_note annotation
        max_stretch: Maximum fret stretch (typically 4 frets)
    
    Returns:
        Modified JAMS object with optimized positions
    """
    # Find tab annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_ann = ann
            break
    
    if not tab_ann or len(tab_ann.data) < 2:
        return jam
    
    print("Optimizing tablature positions for playability...")
    
    # Re-calculate positions considering hand position
    notes = list(tab_ann.data)
    current_position = None  # (center_fret, string)
    
    for i, obs in enumerate(notes):
        pitch = obs.value['pitch']
        
        # Get all possible positions
        positions = midi_pitch_to_guitar_positions(pitch)
        
        if not positions:
            continue
        
        if current_position is None:
            # First note - choose most natural position
            string, fret = positions[0]
            current_position = fret
        else:
            # Choose position within max_stretch of current position
            valid_positions = [
                (s, f) for s, f in positions
                if abs(f - current_position) <= max_stretch
            ]
            
            if valid_positions:
                # Choose closest to current position
                string, fret = min(valid_positions, 
                                  key=lambda p: abs(p[1] - current_position))
            else:
                # No position within reach - need to shift hand position
                string, fret = positions[0]
                current_position = fret
        
        # Update the note
        obs.value['string'] = string
        obs.value['fret'] = fret
    
    print("✓ Positions optimized")
    return jam

In [215]:
def add_random_techniques_to_existing_jam(jam, technique_probability=0.75):
    '''
    Adds random techniques to EXISTING tab_note annotation
    '''
    tech_options = ["vibrato", "hammer-on", "pull-off", "bend", "harmonic"]
    # Find the EXISTING tab_note annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == 'tab_note':
            tab_ann = ann
            break
    
    if tab_ann is None:
        raise ValueError("No tab_note annotation found! Run midi_to_jams_with_tablature() first")
    
    technique_count = 0
    
    # MODIFY existing notes (don't create new ones)
    for obs in tab_ann.data:
        # Add technique to existing note
        if random.random() < technique_probability:
            tech = random.choice(tech_options)
            obs.value['techniques'] = [tech]  # ← Modify existing
            technique_count += 1
        else:
            obs.value['techniques'] = []
    
    print(f"\n✓ Modified {len(tab_ann.data)} notes")
    print(f"✓ Added {technique_count} techniques ({technique_count/len(tab_ann.data)*100:.1f}%)")
    
    return jam

In [216]:
# Function to convert GuitarSet format
def guitarset_jams_to_tab_note(jam):
    """
    Convert GuitarSet JAMS format to include tab_note annotation.
    
    GuitarSet stores note_midi annotations separately for each string (6 total).
    This function combines them into a single tab_note annotation with
    string and fret information.
    """
    STANDARD_TUNING = {
        6: 40,  # E2 (low E)
        5: 45,  # A2
        4: 50,  # D3
        3: 55,  # G3
        2: 59,  # B3
        1: 64   # E4 (high e)
    }
    
    note_midi_annotations = [ann for ann in jam.annotations if ann.namespace == 'note_midi']
    
    tab_ann = jams.Annotation(namespace='pitch_class')
    tab_ann.namespace = 'tab_note'
    
    all_notes = []
    
    for string_idx, ann in enumerate(note_midi_annotations):
        string_num = 6 - string_idx  # index 0 = string 6 (low E)
        open_string_pitch = STANDARD_TUNING[string_num]
        
        for obs in ann.data:
            midi_pitch_rounded = round(obs.value)
            fret = midi_pitch_rounded - open_string_pitch
            fret = max(0, min(24, fret))
            
            all_notes.append({
                'time': obs.time,
                'duration': obs.duration,
                'pitch': midi_pitch_rounded,
                'string': string_num,
                'fret': fret,
                'techniques': [],
                'confidence': obs.confidence if obs.confidence is not None else 1.0
            })
    
    all_notes.sort(key=lambda x: x['time'])
    
    for note_data in all_notes:
        tab_ann.append(
            time=note_data['time'],
            duration=note_data['duration'],
            value={
                'pitch': note_data['pitch'],
                'string': note_data['string'],
                'fret': note_data['fret'],
                'techniques': note_data['techniques']
            },
            confidence=note_data['confidence']
        )
    
    # Extract tempo
    tempo_annotations = [ann for ann in jam.annotations if ann.namespace == 'tempo']
    if tempo_annotations and len(tempo_annotations[0].data) > 0:
        jam.sandbox.tempo_bpm = tempo_annotations[0].data[0].value
    
    # Extract time signature
    beat_annotations = [ann for ann in jam.annotations if ann.namespace == 'beat_position']
    if beat_annotations and len(beat_annotations[0].data) > 0:
        beat_info = beat_annotations[0].data[0].value
        jam.sandbox.time_sig_numerator = beat_info.get('num_beats', 4)
        jam.sandbox.time_sig_denominator = beat_info.get('beat_units', 4)
    
    jam.annotations.append(tab_ann)
    
    print(f"Created tab_note annotation with {len(all_notes)} notes")
    return jam

In [217]:
# Step 1: Load and prep jams for conversion to tabs
jam = jams.load('00_BN1-129-Eb_comp.jams')
jams = guitarset_jams_to_tab_note(jam)

# Step 2: Add techniques to EXISTING tab_note annotation
jam = add_random_techniques_to_existing_jam(jam, technique_probability=0.75)

# Step 3: Debug - check techniques are there
tab_ann = [a for a in jam.annotations if a.namespace == 'tab_note'][0]
print("\nNotes with techniques:")
for i, obs in enumerate(tab_ann.data[:20]):
    techs = obs.value.get('techniques', [])
    if techs:
        print(f"  Note {i}: {techs}")

# Step 4: Convert to MusicXML
bpm = getattr(jam.sandbox, 'tempo_bpm', 120)
jams_to_musicxml_real(jam, 'test_output.musicxml', tempo_bpm=bpm)

Created tab_note annotation with 133 notes

✓ Modified 133 notes
✓ Added 94 techniques (70.7%)

Notes with techniques:
  Note 0: ['harmonic']
  Note 1: ['vibrato']
  Note 2: ['vibrato']
  Note 3: ['vibrato']
  Note 4: ['harmonic']
  Note 5: ['hammer-on']
  Note 6: ['bend']
  Note 10: ['vibrato']
  Note 11: ['pull-off']
  Note 13: ['pull-off']
  Note 14: ['harmonic']
  Note 16: ['harmonic']
  Note 17: ['pull-off']
  Note 18: ['bend']
  Note 19: ['hammer-on']
Converting 133 notes to MusicXML...
  Total duration: 48.00 beats -> 12 measures
Vibrato detected
hammer-on detected at note 2
pull-off detected at note 5
hammer-on detected at note 8
bend detected
Harmonic detected
pull-off detected at note 12
Harmonic detected
hammer-on detected at note 15
Harmonic detected
bend detected
bend detected
bend detected
Harmonic detected
bend detected
bend detected
pull-off detected at note 34
bend detected
hammer-on detected at note 39
hammer-on detected at note 47
bend detected
✓ Created test_output.

'test_output.musicxml'